In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_extraction_gsheet.py
========================
Colab batch runner that works DIRECTLY on a Google Sheet - no more
download-csv / edit-in-excel / re-upload cycle.

How it works:
  - You keep your data in a normal Google Sheet (edit it in the browser any time).
  - Colab authenticates with your Google account (one click, no keys to paste
    besides the Azure one) and reads the rows straight from the sheet.
  - After each link finishes, ONLY that row's graph+status cells are written
    back to the sheet. Your manual edits to other rows are never touched,
    and you can watch statuses appear live while the run is going.

ONE-TIME SETUP (2 minutes):
  1. In Drive, right-click trajectories.csv (or recipes.csv) -> Open with
     -> Google Sheets. Then in that Sheet: File -> Save as Google Sheets.
     This creates the native Sheet you'll use from now on.
  2. Copy the Sheet's URL from the browser bar into SHEET_URL below.
  3. Run this script. A Google popup asks you to allow access - click through.

RULE OF THUMB: don't hand-edit the graph/status columns of rows the runner is
currently processing; anything else you can edit freely, even mid-run.
"""

import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("openai", "openai"), ("requests", "requests"),
                   ("trafilatura", "trafilatura"), ("bs4", "beautifulsoup4"),
                   ("json_repair", "json-repair"), ("gspread", "gspread")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        print(f"Installing {_pkg} ..."); _pip([_pkg])

import json, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests
from openai import OpenAI
import gspread

try:
    import trafilatura; _HAVE_TRAFILATURA = True
except Exception:
    _HAVE_TRAFILATURA = False
try:
    from bs4 import BeautifulSoup; _HAVE_BS4 = True
except Exception:
    _HAVE_BS4 = False
import json_repair

# =====================================================================
# 1. CONFIG - edit these
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_trajectory_extraction.txt"   # must contain {{INTERVIEW_TEXT}}

# The Google Sheet to work on (paste the browser URL of the Sheet)
SHEET_URL      = "https://docs.google.com/spreadsheets/d/1DPLl0KVl_0zuEmHSQZIDigdBp7W3rvR6I9pAp2KY-Ps/edit?gid=0#gid=0"
WORKSHEET_NAME = None            # None = first tab; or e.g. "Sheet1"

# Column layout (0-based indices, same convention as before)
LINK_COL   = 0
GRAPH_COL  = 5                  # where THIS run writes its graph
STATUS_COL = 6                   # where THIS run writes its status
HAS_HEADER = True

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
AZURE_API_KEY   = ""

MAX_WORKERS     = 5
MAX_OUTPUT_TOK  = 8000
MAX_INPUT_CHARS = 45000
HTTP_TIMEOUT    = 30
MAX_RETRIES     = 5
OVERWRITE       = False          # False = skip rows that already have a graph (silently)

SHEET_CELL_LIMIT = 49500         # Google Sheets hard limit is 50,000 chars/cell
PLACEHOLDER = "{{INTERVIEW_TEXT}}"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

# =====================================================================
# 2. GOOGLE AUTH + OPEN SHEET
# =====================================================================
def open_sheet():
    from google.colab import auth as colab_auth  # type: ignore
    colab_auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    if "PASTE_YOUR" in SHEET_URL:
        raise SystemExit("Set SHEET_URL in section 1 to your Google Sheet's URL.")
    sh = gc.open_by_url(SHEET_URL)
    ws = sh.worksheet(WORKSHEET_NAME) if WORKSHEET_NAME else sh.get_worksheet(0)
    print(f"[sheet] opened '{sh.title}' / tab '{ws.title}'")
    return ws

def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        pass

def build_client() -> OpenAI:
    key = (AZURE_API_KEY or "").strip()
    if not key:
        raise SystemExit("AZURE_API_KEY is empty. Paste your Azure API key in section 1.")
    return OpenAI(base_url=AZURE_ENDPOINT, api_key=key)

# =====================================================================
# 3. FETCH + CLEAN PAGE TEXT (direct -> Jina fallback)
# =====================================================================
def _fetch_direct(url):
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    html = resp.text
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    return text

def _fetch_jina(url):
    resp = requests.get("https://r.jina.ai/" + url,
                        headers={"User-Agent": USER_AGENT, "X-Return-Format": "text"},
                        timeout=HTTP_TIMEOUT * 2)
    resp.raise_for_status()
    return resp.text

def fetch_text(url):
    text, err = "", ""
    try:
        text = _fetch_direct(url)
    except Exception as exc:
        err = f"direct: {exc}"
    if len(text.strip()) < 400:
        try:
            jtext = _fetch_jina(url)
            if len(jtext.strip()) > len(text.strip()):
                text = jtext
        except Exception as exc:
            err = (err + " | " if err else "") + f"jina: {exc}"
    if not text.strip():
        return "", f"fetch_error: {err or 'empty page'}"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text).strip()
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]"
    return text, "ok"

# =====================================================================
# 4. CALL THE MODEL
# =====================================================================
def call_model(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(delay); delay = min(delay * 2, 60)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts: {last}")

# =====================================================================
# 5. PARSE MODEL OUTPUT (tolerant, both tag names)
# =====================================================================
def parse_graph(raw):
    block = raw.strip()
    m = re.search(r"<(trajectory_graph|recipe_graph)>(.*?)</\1>", block, re.DOTALL | re.IGNORECASE)
    if m:
        block = m.group(2).strip()
    else:
        m2 = re.search(r"\{.*\}", block, re.DOTALL)
        if m2:
            block = m2.group(0)
    block = re.sub(r"^```(?:json)?|```$", "", block.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(block), "parsed"
    except json.JSONDecodeError as exc:
        try:
            repaired = json_repair.loads(block)
            if isinstance(repaired, (dict, list)) and repaired:
                return repaired, "parsed (repaired)"
        except Exception:
            pass
        return None, f"json_decode_error: {exc}"

# =====================================================================
# 6. SHEET WRITE-BACK (per-row, with 429 retry)
# =====================================================================
def a1_col(idx0):
    """0-based column index -> A1 letter(s)."""
    s, n = "", idx0 + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s

def write_row(ws, sheet_row, graph_val, status_val, lock):
    """Write graph+status cells of one sheet row (1-based row number)."""
    c1, c2 = sorted([GRAPH_COL, STATUS_COL])
    # build a contiguous range covering both cells, preserving anything between
    rng = f"{a1_col(c1)}{sheet_row}:{a1_col(c2)}{sheet_row}"
    values = [""] * (c2 - c1 + 1)
    values[GRAPH_COL - c1] = graph_val
    values[STATUS_COL - c1] = status_val
    delay = 2.0
    for attempt in range(6):
        try:
            with lock:
                if c2 - c1 == 1:      # adjacent columns - single clean update
                    ws.update(rng, [values], value_input_option="RAW")
                else:                  # non-adjacent - two cell updates
                    ws.update_cell(sheet_row, GRAPH_COL + 1, graph_val)
                    ws.update_cell(sheet_row, STATUS_COL + 1, status_val)
            return True
        except Exception as exc:
            if "429" in str(exc) or "quota" in str(exc).lower():
                time.sleep(delay); delay = min(delay * 2, 30)
                continue
            print(f"    !! sheet write failed row {sheet_row}: {str(exc)[:100]}")
            return False
    return False

# =====================================================================
# 7. MAIN
# =====================================================================
def main():
    mount_drive()
    template = Path(PROMPT_FILE).read_text(encoding="utf-8") if Path(PROMPT_FILE).exists() else ""
    if not template:
        raise SystemExit(f"Prompt file not found: {PROMPT_FILE}")
    if PLACEHOLDER not in template:
        raise SystemExit(f"Prompt file must contain {PLACEHOLDER}.")

    ws = open_sheet()
    all_rows = ws.get_all_values()   # list of lists of strings
    if not all_rows:
        raise SystemExit("Sheet is empty.")
    start = 1 if HAS_HEADER else 0
    width = max(LINK_COL, GRAPH_COL, STATUS_COL) + 1
    data = [r + [""] * (width - len(r)) if len(r) < width else r for r in all_rows]

    if HAS_HEADER:
        hdr_updates = []
        if (data[0][GRAPH_COL] or "").strip() != GRAPH_HEADER:
            hdr_updates.append((1, GRAPH_COL + 1, GRAPH_HEADER))
        if (data[0][STATUS_COL] or "").strip() != STATUS_HEADER:
            hdr_updates.append((1, STATUS_COL + 1, STATUS_HEADER))
        for r, c, v in hdr_updates:
            ws.update_cell(r, c, v)

    # decide which rows to process - skipped rows are never written to
    todo, skipped = [], 0
    for i in range(start, len(data)):
        link = (data[i][LINK_COL] or "").strip()
        if not link:
            skipped += 1; continue
        if (data[i][GRAPH_COL] or "").strip() and not OVERWRITE:
            skipped += 1; continue
        todo.append(i)

    client = build_client()
    print(f"\n[run] {len(todo)} link(s) to process | {skipped} skipped untouched "
          f"| workers={MAX_WORKERS} | model={DEPLOYMENT_NAME}")
    print(f"[run] graph -> col {a1_col(GRAPH_COL)} ('{GRAPH_HEADER}') | "
          f"status -> col {a1_col(STATUS_COL)} ('{STATUS_HEADER}')\n")

    counts, counts_lock, sheet_lock, done = {}, threading.Lock(), threading.Lock(), 0

    def work(idx):
        link = (data[idx][LINK_COL] or "").strip()
        sheet_row = idx + 1                      # sheets are 1-based
        text, fnote = fetch_text(link)
        if not text:
            write_row(ws, sheet_row, "", f"fetch_failed: {fnote}"[:400], sheet_lock)
            return idx, "fetch_failed"
        try:
            raw = call_model(client, template.replace(PLACEHOLDER, text))
        except Exception as exc:
            write_row(ws, sheet_row, "", f"api_failed: {str(exc)[:150]}", sheet_lock)
            return idx, "api_failed"
        graph, pnote = parse_graph(raw)
        if graph is None:
            write_row(ws, sheet_row, raw.strip()[:SHEET_CELL_LIMIT],
                      f"parse_failed: {pnote}"[:400], sheet_lock)
            return idx, "parse_failed"
        blob = json.dumps(graph, ensure_ascii=False)
        n_nodes = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
        if len(blob) > SHEET_CELL_LIMIT:
            # graph too big for one cell -> save full JSON to Drive, keep pointer in cell
            overflow_dir = Path(DRIVE_FOLDER) / "graphs_overflow"
            overflow_dir.mkdir(exist_ok=True)
            fpath = overflow_dir / f"row{sheet_row}.json"
            fpath.write_text(blob, encoding="utf-8")
            write_row(ws, sheet_row, f"@file:{fpath.name}",
                      f"ok ({n_nodes} nodes) [stored in graphs_overflow/{fpath.name} - "
                      f"{len(blob)} chars exceeds sheet cell limit]", sheet_lock)
            return idx, "ok_overflow"
        status = f"ok ({n_nodes} nodes)" + (" [repaired]" if "repaired" in pnote else "")
        write_row(ws, sheet_row, blob, status, sheet_lock)
        return idx, "ok"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(work, i): i for i in todo}
        for fut in as_completed(futures):
            idx, status = fut.result()
            with counts_lock:
                counts[status] = counts.get(status, 0) + 1
                done += 1
            print(f"[{done}/{len(todo)}] {status:12s} {(data[idx][LINK_COL] or '')[:80]}")

    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print("[done] results are already in the Google Sheet - nothing to download/upload.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[sheet] opened 'recipes_final' / tab 'Sheet1'

[run] 1 link(s) to process | 59 skipped untouched | workers=5 | model=gpt-5.1
[run] graph -> col F ('recipe_graph') | status -> col G ('recipe_status')



/tmp/ipykernel_688/1488001006.py:237: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(rng, [values], value_input_option="RAW")


[1/1] ok           https://review.firstround.com/the-founder-dating-playbook-heres-the-process-i-us

[done] ok=1
[done] results are already in the Google Sheet - nothing to download/upload.


In [ ]:
# ============================================================
# RETRY CELL - re-process ONLY the failed rows via Jina Reader
# Self-contained: paste and run as its own Colab cell.
# Finds rows whose graph cell is empty (fetch_failed / api_failed /
# parse_failed / thin), fetches them through r.jina.ai, calls GPT-5.1,
# and writes the graph back into the same CSV.
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "openai", "requests", "json-repair"], check=False)

import csv, json, re, time
from pathlib import Path
import requests, json_repair
from openai import OpenAI
from google.colab import drive
drive.mount('/content/drive')

# ---- config (must match the main runner) ----
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_trajectory_extraction.txt"
LINKS_FILE   = f"{DRIVE_FOLDER}/trajectories.csv"
LINK_COL, GRAPH_COL, STATUS_COL, HAS_HEADER = 0, 3, 4, True

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
AZURE_API_KEY   = ""


MAX_OUTPUT_TOK, MAX_INPUT_CHARS, HTTP_TIMEOUT, MAX_RETRIES = 8000, 45000, 60, 5
PLACEHOLDER = "{{INTERVIEW_TEXT}}"
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0",
      "X-Return-Format": "text"}

assert AZURE_API_KEY.strip(), "Paste your Azure API key into AZURE_API_KEY first."
client   = OpenAI(base_url=AZURE_ENDPOINT, api_key=AZURE_API_KEY.strip())
template = Path(PROMPT_FILE).read_text(encoding="utf-8")
assert PLACEHOLDER in template

# ---- load CSV ----
with open(LINKS_FILE, newline="", encoding="utf-8-sig") as fh:
    rows = [list(r) for r in csv.reader(fh)]
header, data = (rows[0], rows[1:]) if HAS_HEADER else (None, rows)
for r in data:
    while len(r) <= max(GRAPH_COL, STATUS_COL): r.append("")

def save():
    with open(LINKS_FILE, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        if header: w.writerow(header)
        w.writerows(data)

# ---- select ONLY failed rows: link present, graph cell empty, status not ok ----
failed = [i for i, r in enumerate(data)
          if (r[LINK_COL] or "").strip()
          and not (r[GRAPH_COL] or "").strip()
          and not str(r[STATUS_COL]).startswith("ok")]
print(f"failed rows to retry: {len(failed)}")
for i in failed:
    print("  -", data[i][STATUS_COL][:50], "|", data[i][LINK_COL][:70])

# ---- Jina fetch + model call + parse ----
def fetch_jina(url):
    r = requests.get("https://r.jina.ai/" + url, headers=UA, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    t = re.sub(r"\n\s*\n\s*\n+", "\n\n", re.sub(r"[ \t]+", " ", r.text)).strip()
    return t[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]" if MAX_INPUT_CHARS and len(t) > MAX_INPUT_CHARS else t

def call_model(prompt):
    delay = 4.0
    for a in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK)
            return resp.choices[0].message.content or ""
        except Exception as e:
            if a == MAX_RETRIES - 1: raise
            time.sleep(delay); delay = min(delay * 2, 60)

def parse_graph(raw):
    b = raw.strip()
    m = re.search(r"<trajectory_graph>(.*?)</trajectory_graph>", b, re.S | re.I)
    b = m.group(1).strip() if m else (re.search(r"\{.*\}", b, re.S).group(0) if re.search(r"\{.*\}", b, re.S) else b)
    b = re.sub(r"^```(?:json)?|```$", "", b, flags=re.M).strip()
    try:
        return json.loads(b)
    except Exception:
        g = json_repair.loads(b)
        return g if isinstance(g, dict) and g else None

# ---- run the retries (sequential - only a handful of rows) ----
results = {}
for n, i in enumerate(failed, 1):
    row, link = data[i], data[i][LINK_COL].strip()
    try:
        text = fetch_jina(link)
        if len(text) < 400:
            row[STATUS_COL] = f"jina_thin ({len(text)} chars)"; results["jina_thin"] = results.get("jina_thin",0)+1
            print(f"[{n}/{len(failed)}] jina_thin    {link[:70]}"); save(); continue
    except Exception as e:
        row[STATUS_COL] = f"jina_failed: {str(e)[:60]}"; results["jina_failed"] = results.get("jina_failed",0)+1
        print(f"[{n}/{len(failed)}] jina_failed  {link[:70]}"); save(); continue
    try:
        raw = call_model(template.replace(PLACEHOLDER, text))
    except Exception:
        row[STATUS_COL] = "api_failed"; results["api_failed"] = results.get("api_failed",0)+1
        print(f"[{n}/{len(failed)}] api_failed   {link[:70]}"); save(); continue
    g = parse_graph(raw)
    if g is None:
        row[GRAPH_COL] = raw.strip(); row[STATUS_COL] = "parse_failed"
        results["parse_failed"] = results.get("parse_failed",0)+1
        print(f"[{n}/{len(failed)}] parse_failed {link[:70]}")
    else:
        row[GRAPH_COL] = json.dumps(g, ensure_ascii=False)
        row[STATUS_COL] = f"ok ({len(g.get('nodes',[]))} nodes) [jina]"
        results["ok"] = results.get("ok",0)+1
        print(f"[{n}/{len(failed)}] ok           {link[:70]}")
    save()

print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(results.items())))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
failed rows to retry: 6
  - fetch_failed: fetch_error: 403 Client Error: Forbi | https://www.findaphd.com/blog/5835/finding-the-one-my-phd-application-
  - fetch_failed: fetch_error: 403 Client Error: Forbi | https://www.dementiaresearcher.nihr.ac.uk/blog-journey-to-a-phd-how-to
  - fetch_failed: fetch_error: 403 Client Error: Forbi | https://www.dementiaresearcher.nihr.ac.uk/blog-the-long-way-round-to-a
  - fetch_failed: fetch_error: 404 Client Error: Not F | https://blog.phusion.nl/2016/10/04/bootstrapped-profitable-and-proud/
  - fetch_failed: fetch_error: 403 Client Error: Forbi | https://medium.com/inside-wattage/well-we-failed-77e795e16ecf
  - fetch_failed: fetch_error: HTTPSConnectionPool(hos | https://www.coderholic.com/my-startup-failed-but-its-ok/
[1/6] ok           https://www.findaphd.com/blog/5835/finding-the-one-my-phd-application-
[2/6] ok     

In [ ]:
# ============================================================
# DESCRIPTIVE ANALYSIS - both banks (trajectories.csv + recipes.csv)
# Matches the batch-runner schema: col1 link | col2 title | col3 domain |
# col4 graph JSON | col5 graph_status ("ok (N nodes)...").
# Adds descriptive columns per row, saves *_with_descriptives.csv,
# prints per-file summaries + a side-by-side comparison of the two banks.
# Paste as one Colab cell and run.
# ============================================================
import os, json, ast, re
import pandas as pd
from collections import Counter, defaultdict
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
FILES = {"trajectory": f"{DRIVE_FOLDER}/trajectories.csv",
         "recipe":     f"{DRIVE_FOLDER}/recipes.csv"}
GRAPH_COL_IDX, STATUS_COL_IDX = 3, 4          # positional, matching the runner

# ---------------- graph cell parsing ----------------
def parse_graph(cell):
    if pd.isna(cell) or not str(cell).strip():
        return None, "empty"
    s = re.sub(r"^```(?:json)?|```$", "", str(cell).strip(), flags=re.M).strip()
    for loader in (json.loads, ast.literal_eval):
        try:
            g = loader(s)
            if isinstance(g, dict):
                return g, "ok"
        except Exception:
            continue
    return None, "parse_error"

# ---------------- per-graph descriptives ----------------
def describe(g):
    nodes = g.get("nodes", []) or []
    edges = g.get("edges", []) or []
    ids = [n.get("id") for n in nodes]; idset = set(ids)
    adj = defaultdict(list); indeg, outdeg = Counter(), Counter(); dangling = 0
    for e in edges:
        s = e.get("source", e.get("from")); t = e.get("target", e.get("to"))
        if s in idset and t in idset:
            adj[s].append(t); outdeg[s] += 1; indeg[t] += 1
        else:
            dangling += 1
    # cycle detection (iterative DFS)
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {i: WHITE for i in idset}; cycle = False
    for start in idset:
        if color[start] != WHITE: continue
        stack = [(start, iter(adj[start]))]; color[start] = GRAY
        while stack and not cycle:
            u, it = stack[-1]; nxt = next(it, None)
            if nxt is None: color[u] = BLACK; stack.pop()
            elif color[nxt] == GRAY: cycle = True
            elif color[nxt] == WHITE: color[nxt] = GRAY; stack.append((nxt, iter(adj[nxt])))
    longest = None
    if not cycle and idset:
        order, seen = [], set()
        def topo(u):
            seen.add(u)
            for v in adj[u]:
                if v not in seen: topo(v)
            order.append(u)
        for i in idset:
            if i not in seen: topo(i)
        dist = {i: 0 for i in idset}
        for u in reversed(order):
            for v in adj[u]:
                dist[v] = max(dist[v], dist[u] + 1)
        longest = max(dist.values()) if dist else 0
    n = len(nodes)
    return {"n_nodes": n, "n_edges": len(edges), "has_cycle": cycle,
            "longest_path_edges": longest,
            "density": round(len(edges)/(n*(n-1)), 4) if n > 1 else 0.0,
            "linearity": round(longest/len(edges), 3) if longest and edges else None,
            "max_out_degree": max(outdeg.values()) if outdeg else 0,
            "n_roots": sum(1 for i in idset if indeg[i] == 0),
            "n_leaves": sum(1 for i in idset if outdeg[i] == 0),
            "isolated_nodes": sum(1 for i in idset if indeg[i]==0 and outdeg[i]==0),
            "dangling_edges": dangling,
            "duplicate_node_ids": len(ids) - len(idset),
            "evidence_coverage": round(sum(1 for x in nodes
                                           if str(x.get("evidence","")).strip())/n, 3) if n else 0.0,
            "node_type_counts": json.dumps(dict(Counter(str(x.get("type","?")) for x in nodes)))}

DESC = ["n_nodes","n_edges","has_cycle","longest_path_edges","density","linearity",
        "max_out_degree","n_roots","n_leaves","isolated_nodes","dangling_edges",
        "duplicate_node_ids","evidence_coverage","node_type_counts","parse_status"]

# ---------------- process both banks ----------------
banks = {}
for kind, path in FILES.items():
    if not os.path.exists(path):
        print(f"!! {path} not found, skipping"); continue
    df = pd.read_csv(path)
    gcol, scol = df.columns[GRAPH_COL_IDX], df.columns[STATUS_COL_IDX]
    recs = []
    for _, row in df.iterrows():
        status = str(row.get(scol, ""))
        if not status.startswith("ok"):                     # only extracted rows
            recs.append({**{c: None for c in DESC[:-1]}, "parse_status": f"skipped ({status[:25] or 'no status'})"})
            continue
        g, ps = parse_graph(row.get(gcol))
        d = describe(g) if g else {c: None for c in DESC[:-1]}
        d["parse_status"] = ps
        recs.append(d)
    df = pd.concat([df.reset_index(drop=True), pd.DataFrame(recs)], axis=1)
    out = path.replace(".csv", "_with_descriptives.csv")
    df.to_csv(out, index=False)
    banks[kind] = df

    ok = df[df.parse_status == "ok"]
    print(f"\n================ {kind.upper()} bank ({os.path.basename(path)}) ================")
    print(f"rows: {len(df)} | graphs analyzed: {len(ok)} | skipped/not-extracted: {len(df)-len(ok)}")
    if len(ok):
        print(ok[["n_nodes","n_edges","longest_path_edges","density","linearity","evidence_coverage"]]
              .describe().round(2).loc[["mean","50%","std","min","max"]])
        cyc = ok[ok.has_cycle == True]
        print(f"CYCLE VERIFICATION: {len(cyc)} graph(s) with cycles"
              + ("  <-- inspect:" if len(cyc) else "  (verified: all acyclic)"))
        if len(cyc):
            cols = [c for c in df.columns[:2]]
            print(cyc[cols + ["n_nodes","n_edges"]].to_string(index=False))
        dom = df.columns[2]
        print("per-domain medians:")
        print(ok.groupby(dom)[["n_nodes","n_edges","linearity"]].median().round(2))
    print(f"saved -> {out}")

# ---------------- side-by-side comparison ----------------
if len(banks) == 2:
    print("\n================ TRAJECTORY vs RECIPE (medians) ================")
    comp = {}
    for kind, df in banks.items():
        ok = df[df.parse_status == "ok"]
        comp[kind] = ok[["n_nodes","n_edges","longest_path_edges","linearity",
                         "max_out_degree","evidence_coverage"]].median().round(2)
    print(pd.DataFrame(comp))
    print("\nExpectation check: trajectories should show HIGHER linearity and LOWER "
          "max_out_degree than recipes. If not, the extractor may be blurring the types.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

================ TRAJECTORY bank (trajectories.csv) ================
rows: 60 | graphs analyzed: 59 | skipped/not-extracted: 1
      n_nodes  n_edges  longest_path_edges  density  linearity  \
mean    31.03    29.92               23.57     0.04       0.79   
50%     30.00    30.00               22.00     0.03       0.92   
std     12.23    12.11               12.95     0.01       0.24   
min      0.00     0.00                6.00     0.00       0.13   
max     71.00    70.00               70.00     0.08       1.00   

      evidence_coverage  
mean               0.98  
50%                1.00  
std                0.13  
min                0.00  
max                1.00  
CYCLE VERIFICATION: 9 graph(s) with cycles  <-- inspect:
                                                                                                                              link   

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_recipes_on_trajectories_colab.py
====================================
RERUN recipe-graph extraction on trajectories.csv, writing results NEXT TO the
trajectory columns. Final layout:
  link(0) | short_title(1) | domain(2) | trajectory_graph(3) | graph_status(4)
  | recipe_graph(5) | recipe_status(6)
OVERWRITE=True: every row is re-extracted, so recipe_status gets a proper
"ok (N nodes)" per row - replacing the old "already_done" junk in column F and
the stale recipe graphs in column G.

FIXES vs the previous version:
  - Skipped rows are LEFT UNTOUCHED: no more "already_done" being written
    into (and polluting) the status column. The status column for the second
    graph now only ever contains real outcomes: ok (N nodes) / fetch_failed /
    api_failed / parse_failed - exactly like the first extraction run.
  - The graph/status column HEADERS are explicit config (no more duplicate
    "graph_status" headers).
  - OVERWRITE back to False: rows that already have a second graph are skipped
    silently; set True only to deliberately redo them.
  - Jina Reader fallback included for pages that fail direct fetch.

CSV layout is positional and configurable below. For recipes.csv today:
  link(0) | short_title(1) | domain(2) | recipe_graph(3) | graph_status(4)
  | (5 unused) | trajectory_graph(6) | trajectory_status(7)

AUTHENTICATION: API key only - paste into AZURE_API_KEY.
"""

# =====================================================================
# 0. DEPENDENCIES
# =====================================================================
import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("openai", "openai"), ("requests", "requests"),
                   ("trafilatura", "trafilatura"), ("bs4", "beautifulsoup4"),
                   ("json_repair", "json-repair")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        print(f"Installing {_pkg} ..."); _pip([_pkg])

import csv, json, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests
from openai import OpenAI

try:
    import trafilatura; _HAVE_TRAFILATURA = True
except Exception:
    _HAVE_TRAFILATURA = False
try:
    from bs4 import BeautifulSoup; _HAVE_BS4 = True
except Exception:
    _HAVE_BS4 = False
import json_repair

# =====================================================================
# 1. CONFIG - edit these
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"

# Which extraction is this run? (prompt + where its output goes)
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_recipe_extraction.txt"   # must contain {{INTERVIEW_TEXT}}
LINKS_FILE   = f"{DRIVE_FOLDER}/trajectories.csv"
OUTPUT_CSV   = LINKS_FILE

LINK_COL      = 0
GRAPH_COL     = 5                     # F: recipe graph, right after trajectory graph_status
STATUS_COL    = 6                     # G: recipe status "ok (N nodes)" - overwrites stale data there
GRAPH_HEADER  = "recipe_graph"        # header names written for those columns
STATUS_HEADER = "recipe_status"
HAS_HEADER    = True

# Example for the mirror run (recipe graphs into trajectories.csv):
#   PROMPT_FILE = f"{DRIVE_FOLDER}/prompt_recipe_extraction.txt"
#   LINKS_FILE  = f"{DRIVE_FOLDER}/trajectories.csv"
#   GRAPH_COL, STATUS_COL = 6, 5        (matching its current layout)
#   GRAPH_HEADER, STATUS_HEADER = "recipe_graph", "recipe_status"

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
AZURE_API_KEY   = ""

MAX_WORKERS     = 5
MAX_OUTPUT_TOK  = 8000
MAX_INPUT_CHARS = 45000
HTTP_TIMEOUT    = 30
MAX_RETRIES     = 5
OVERWRITE       = True    # RERUN everything so recipe_status is properly produced per row

PLACEHOLDER = "{{INTERVIEW_TEXT}}"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

# =====================================================================
# 2. MOUNT DRIVE
# =====================================================================
def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        print("[drive] mounted")
    except Exception as exc:
        print(f"[drive] not mounting (not on Colab or already mounted): {exc}")

# =====================================================================
# 3. AZURE CLIENT
# =====================================================================
def build_client() -> OpenAI:
    key = (AZURE_API_KEY or "").strip()
    if not key:
        raise SystemExit("AZURE_API_KEY is empty. Paste your Azure API key in section 1 and re-run.")
    return OpenAI(base_url=AZURE_ENDPOINT, api_key=key)

# =====================================================================
# 4. READ CSV
# =====================================================================
def read_table(path_str):
    path = Path(path_str)
    if not path.exists():
        raise SystemExit(f"Links file not found: {path}")
    delim = "\t" if path.suffix.lower() == ".tsv" else ","
    with path.open(newline="", encoding="utf-8-sig") as fh:
        all_rows = [list(r) for r in csv.reader(fh, delimiter=delim)]
    if not all_rows:
        raise SystemExit(f"{path} is empty.")
    return (all_rows[0], all_rows[1:]) if HAS_HEADER else (None, all_rows)

def pad(row, width):
    if len(row) < width:
        row.extend([""] * (width - len(row)))
    return row

# =====================================================================
# 5. FETCH + CLEAN PAGE TEXT  (direct -> Jina fallback)
# =====================================================================
def _fetch_direct(url):
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    html = resp.text
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    return text

def _fetch_jina(url):
    resp = requests.get("https://r.jina.ai/" + url,
                        headers={"User-Agent": USER_AGENT, "X-Return-Format": "text"},
                        timeout=HTTP_TIMEOUT * 2)
    resp.raise_for_status()
    return resp.text

def fetch_text(url):
    text, source, err = "", "", ""
    try:
        text = _fetch_direct(url); source = "direct"
    except Exception as exc:
        err = f"direct: {exc}"
    if len(text.strip()) < 400:
        try:
            jtext = _fetch_jina(url)
            if len(jtext.strip()) > len(text.strip()):
                text, source = jtext, "jina"
        except Exception as exc:
            err = (err + " | " if err else "") + f"jina: {exc}"
    if not text.strip():
        return "", f"fetch_error: {err or 'empty page'}"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text).strip()
    note = f"ok ({source})"
    if len(text) < 400:
        note = f"thin_text ({len(text)} chars, {source})"
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]"
    return text, note

# =====================================================================
# 6. CALL THE MODEL
# =====================================================================
def call_model(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(delay); delay = min(delay * 2, 60)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts: {last}")

# =====================================================================
# 7. PARSE MODEL OUTPUT -> graph JSON (tolerant, both tag names)
# =====================================================================
def parse_graph(raw):
    block = raw.strip()
    m = re.search(r"<(trajectory_graph|recipe_graph)>(.*?)</\1>", block, re.DOTALL | re.IGNORECASE)
    if m:
        block = m.group(2).strip()
    else:
        m2 = re.search(r"\{.*\}", block, re.DOTALL)
        if m2:
            block = m2.group(0)
    block = re.sub(r"^```(?:json)?|```$", "", block.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(block), "parsed"
    except json.JSONDecodeError as exc:
        try:
            repaired = json_repair.loads(block)
            if isinstance(repaired, (dict, list)) and repaired:
                return repaired, "parsed (repaired)"
        except Exception:
            pass
        return None, f"json_decode_error: {exc}"

# =====================================================================
# 8. MAIN
# =====================================================================
def main():
    mount_drive()

    template = Path(PROMPT_FILE).read_text(encoding="utf-8") if Path(PROMPT_FILE).exists() else ""
    if not template:
        raise SystemExit(f"Prompt file not found: {PROMPT_FILE}")
    if PLACEHOLDER not in template:
        raise SystemExit(f"Prompt file must contain {PLACEHOLDER}.")

    header, data = read_table(LINKS_FILE)
    if not data:
        raise SystemExit("No data rows to process.")

    width = max(LINK_COL, GRAPH_COL, STATUS_COL if STATUS_COL is not None else 0) + 1
    for row in data:
        pad(row, width)
    if header is not None:
        pad(header, width)
        header[GRAPH_COL] = GRAPH_HEADER          # explicit headers - no duplicates
        if STATUS_COL is not None:
            header[STATUS_COL] = STATUS_HEADER

    src = Path(LINKS_FILE)
    out_csv = Path(OUTPUT_CSV) if OUTPUT_CSV else src.with_name(f"{src.stem}_with_graphs.csv")
    save_lock = threading.Lock()

    def save():
        with save_lock:
            with out_csv.open("w", newline="", encoding="utf-8") as fh:
                w = csv.writer(fh)
                if header is not None:
                    w.writerow(header)
                w.writerows(data)

    def set_status(row, val):
        if STATUS_COL is not None:
            row[STATUS_COL] = val

    # decide which rows to process - skipped rows are NOT modified in any way
    todo, skipped = [], 0
    for idx, row in enumerate(data):
        link = (row[LINK_COL] or "").strip()
        if not link:
            skipped += 1
            continue
        if (row[GRAPH_COL] or "").strip() and not OVERWRITE:
            skipped += 1                      # leave existing graph AND status untouched
            continue
        todo.append(idx)

    client = build_client()
    print(f"\n[run] {len(todo)} link(s) to process | {skipped} skipped untouched "
          f"| workers={MAX_WORKERS} | model={DEPLOYMENT_NAME}")
    print(f"[run] graph -> col {GRAPH_COL + 1} ('{GRAPH_HEADER}') | "
          f"status -> col {STATUS_COL + 1} ('{STATUS_HEADER}') | output -> {out_csv}\n")

    counts, counts_lock, done = {}, threading.Lock(), 0

    def work(idx):
        row = data[idx]
        link = (row[LINK_COL] or "").strip()
        text, fnote = fetch_text(link)
        if not text:
            row[GRAPH_COL] = ""
            set_status(row, f"fetch_failed: {fnote}")
            return idx, "fetch_failed"
        try:
            raw = call_model(client, template.replace(PLACEHOLDER, text))
        except Exception:
            row[GRAPH_COL] = ""
            set_status(row, "api_failed")
            return idx, "api_failed"
        graph, pnote = parse_graph(raw)
        if graph is None:
            row[GRAPH_COL] = raw.strip()
            set_status(row, f"parse_failed: {pnote}")
            return idx, "parse_failed"
        row[GRAPH_COL] = json.dumps(graph, ensure_ascii=False)
        n_nodes = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
        set_status(row, f"ok ({n_nodes} nodes)" + (" [repaired]" if "repaired" in pnote else ""))
        return idx, "ok"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(work, i): i for i in todo}
        for fut in as_completed(futures):
            idx, status = fut.result()
            with counts_lock:
                counts[status] = counts.get(status, 0) + 1
                done += 1
            print(f"[{done}/{len(todo)}] {status:12s} {(data[idx][LINK_COL] or '')[:80]}")
            save()

    save()
    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print(f"[done] CSV -> {out_csv}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[drive] mounted

[run] 60 link(s) to process | 0 skipped untouched | workers=5 | model=gpt-5.1
[run] graph -> col 6 ('recipe_graph') | status -> col 7 ('recipe_status') | output -> /content/drive/MyDrive/QCRI_Project/Robee_Dataset/trajectories.csv

[1/60] ok           https://bennokrojer.github.io/far_away.html
[2/60] ok           https://towardsdatascience.com/my-journey-to-phd-admission-in-artificial-intelli
[3/60] ok           https://blog.clementfung.me/ubc-recap
[4/60] ok           https://www.beren.io/2023-04-09-My-PhD-experience/
[5/60] ok           https://thephdplace.com/prep-a-non-traditional-route-to-phd/
[6/60] ok           https://thinktown.medium.com/alumna-story-from-cmu-engineering-student-to-mit-ph
[7/60] ok           https://stevenkolawole.medium.com/the-road-to-phd-ii-my-approach-to-the-applicat
[8/60] fetch_failed https://theinterviewporta

In [ ]:
# ============================================================
# FIX COLUMNS - trajectories.csv (no API calls, instant)
# Current:  D trajectory_graph | E graph_status | F "already_done" junk | G recipe_graph
# Target:   D trajectory_graph | E graph_status | F recipe_graph        | G recipe_status
# The recipe graphs already exist in G - this moves them to F and
# computes a proper "ok (N nodes)" status into G by parsing each graph.
# A backup copy is saved first.
# ============================================================
import csv, json, re, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

FILE = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset/trajectories.csv"
OLD_GRAPH_COL = 6   # G: where the recipe graphs currently sit
NEW_GRAPH_COL = 5   # F: where they should be
NEW_STATUS_COL = 6  # G: where the computed status goes

try:
    import json_repair
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "json-repair"], check=False)
    import json_repair

shutil.copy(FILE, FILE.replace(".csv", "_backup_before_fix.csv"))

with open(FILE, newline="", encoding="utf-8-sig") as fh:
    rows = [list(r) for r in csv.reader(fh)]
header, data = rows[0], rows[1:]

width = max(OLD_GRAPH_COL, NEW_GRAPH_COL, NEW_STATUS_COL) + 1
for r in [header] + data:
    while len(r) < width: r.append("")

header[NEW_GRAPH_COL]  = "recipe_graph"
header[NEW_STATUS_COL] = "recipe_status"

def parse(cell):
    s = re.sub(r"^```(?:json)?|```$", "", str(cell or "").strip(), flags=re.M).strip()
    if not s or not s.startswith("{"):
        return None
    try:
        g = json.loads(s)
    except Exception:
        try:
            g = json_repair.loads(s)
        except Exception:
            return None
    return g if isinstance(g, dict) else None

stats = {"ok": 0, "empty": 0, "parse_failed": 0}
for r in data:
    raw = r[OLD_GRAPH_COL]
    g = parse(raw)
    if g and g.get("nodes"):
        r[NEW_GRAPH_COL]  = json.dumps(g, ensure_ascii=False)
        r[NEW_STATUS_COL] = f"ok ({len(g['nodes'])} nodes)"
        stats["ok"] += 1
    elif not str(raw or "").strip():
        r[NEW_GRAPH_COL]  = ""
        r[NEW_STATUS_COL] = "missing"
        stats["empty"] += 1
    else:
        r[NEW_GRAPH_COL]  = str(raw)          # keep raw so nothing is lost
        r[NEW_STATUS_COL] = "parse_failed"
        stats["parse_failed"] += 1

with open(FILE, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(header)
    w.writerows(data)

print(f"rows: {len(data)} | ok: {stats['ok']} | missing: {stats['empty']} | parse_failed: {stats['parse_failed']}")
print("layout now: D trajectory_graph | E graph_status | F recipe_graph | G recipe_status")
print("backup saved as trajectories_backup_before_fix.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rows: 60 | ok: 0 | missing: 0 | parse_failed: 60
layout now: D trajectory_graph | E graph_status | F recipe_graph | G recipe_status
backup saved as trajectories_backup_before_fix.csv
